# Загружаем данные

Загружаем граф из файла, получая список рёбер, множество вершин и список смежности.

In [1]:
def load_graph(file, directed=False):
    edges = []
    nodes = set()
    adjacency = {}

    with open(file, 'r') as file:
        for line in file:
            # skip comments and empty lines
            if line.startswith('#') or not line.strip():
                continue
            u, v = map(int, line.strip().split())

            edges.append((u, v))
            nodes.update([u, v]) # add both nodes to the set

            if u not in adjacency:
                adjacency[u] = set()
            adjacency[u].add(v)

            if not directed:
                if v not in adjacency:
                    adjacency[v] = set()
                adjacency[v].add(u)

    return edges, nodes, adjacency

# Вычисление расстояний между вершинами сети

## landmarks_basic

Для начала сделаем функцию, которая будет из всех вершин будет выбирать ориентиры. В самом простом случае берём рандомные k вершин:

In [2]:
import random

def select_landmarks(nodes, k):
    len_nodes = len(nodes)
    if k > len_nodes:
        raise ValueError(f"you provided k={k} landmarks, but the graph has only {len_nodes} nodes")
    return random.sample(list(nodes), k)

Затем сделаем функцию для BFS(source vertex), которая по списку смежности будет определять словарь distances. В нём distance[v] -> расстояние от source до v.

In [3]:
from collections import deque

def run_bfs_n_return_distances(adjacency, source):
    distances = {source: 0}
    queue = deque([source])

    while queue: # is not empty
        u = queue.popleft()
        adjacency_u = adjacency.get(u, [])
        for v in adjacency_u:
            if v not in distances:
                distances[v] = distances[u] + 1
                queue.append(v)
    return distances

Далее сделаем функцию, которая будет запускать BFS для всех ориентиров и заполнять словарь landmark_distances. В нём landmark_distances[l] -> distances (словарь из пред. пункта) от l до всех остальных вершин:

In [4]:
def compute_landmark_distances(adjacency, landmarks):
    landmark_distances = {}
    for cur_landmark in landmarks:
        landmark_distances[cur_landmark] = run_bfs_n_return_distances(adjacency, source=cur_landmark)
    return landmark_distances

После этого сделаем функцию, которая будет оценивать расстояние между вершинами s и t по формуле:
$$
d(s, t) \approx \min\limits_{l \in L} (ld[l][s] + ld[l][t])
$$

In [5]:
def estimate_distance(s, t, landmark_distances):
    all_estimates = []
    for l, ld_l in landmark_distances.items():
        ds = ld_l.get(s, float('inf'))
        dt = ld_l.get(t, float('inf'))
        all_estimates.append(ds + dt)
    ans = min(all_estimates) if all_estimates else float('inf')
    return ans

Наконец, можем сделать финальную функцию landmarks_basic, которая использует все предыдущие функции-помощники: среди всех вершин выделяет 'k' ориентиров, вычисляет расстояния от каждого ориентира до всех остальных вершин графа, а затем считает расстояние между двумя вершинами по формуле из предыдущего пункта.

In [6]:
def landmarks_basic(adjacency, nodes, s, t, k):
    landmarks = select_landmarks(nodes, k)
    landmark_distances = compute_landmark_distances(adjacency, landmarks)
    distance = estimate_distance(s, t, landmark_distances)
    return distance

## landmarks_sc

Для начала сделаем функцию, которая заполняет $p_u[v]$ из статьи. Мы назовём возвращаемый ей словарь spt: {v: next_vertex_in_shortest_path_to_l}.

In [7]:
def build_spt(adjacency, landmark):
    spt = {landmark: None}
    visited = {landmark}
    queue = deque([landmark])

    while queue:
        curr = queue.popleft()
        for nbr in adjacency.get(curr, ()):
            if nbr not in visited:
                visited.add(nbr)
                spt[nbr] = curr
                queue.append(nbr)

    return spt

Затем сделаем функцию get_path(s, spt, target_path), которая будет возвращать путь от s до t (t -- вершина из target_path) по словарю spt. Она будет возвращать список вершин, которые лежат на пути от s до t.

In [8]:
def get_path(s, target_path, spt):
    target_path_set = set(target_path) # so membership test is O(1)

    path = [s]
    while s not in target_path_set:
        s = spt[s]
        path.append(s)
    return path

После этого сделаем функцию distance_sc, которая будет возвращать расстояние между двумя вершинами s и c, используя понятие LCA и shortcut'ов из статьи.

In [9]:
def distance_sc(s, t, adjacency, spt_u, u):
    # π1: path from s up to landmark u
    pi1 = get_path(s, {u}, spt_u)        # [s, …, u]

    # π2: path from t up to any node in π1; ends at LCA
    pi2 = get_path(t, pi1, spt_u)        # [t, …, LCA]
    lca = pi2[-1]

    # π3: path from s up to that LCA
    pi3 = get_path(s, {lca}, spt_u)      # [s, …, LCA]

    # base cost: go t --> LCA and s --> LCA without shortcut
    best = (len(pi2) - 1) + (len(pi3) - 1)

    # Try every edge between π2 and π3 to see if it gives a shorter path
    for i, w in enumerate(pi2):
        for j, w_prime in enumerate(pi3):
            if w_prime in adjacency.get(w, ()):
                curr = i + 1 + j
                if curr < best:
                    best = curr

    return best

Наконец, сделаем функцию landmarks_sc, которая будет использовать все предыдущие функции-помощники: выбирает k ориентиров, для каждого из них вычисляет дерево spt, а затем использует его для вычисления расстояний между парами вершин, используя эвристики (LCA и shortcut'ы). 

In [10]:
def landmarks_sc(adjacency, nodes, s, t, k):
    best = float('inf')

    for u in select_landmarks(nodes, k):
        spt_u = build_spt(adjacency, u)

        # if either s or t isn't reachable from u, skip this landmark
        if s not in spt_u or t not in spt_u:
            continue

        # otherwise compute the SC‐distance
        est = distance_sc(s, t, adjacency, spt_u, u)
        best = min(best, est)

    return best


## Исследование работы алгоритмов

Давайте проверим работу этого алгоритма на разных графах.

Определим мета-параметры:

In [11]:
graph_path = "datasets/undirected/CA-AstroPh.txt"
k = 10
# distance_estimator_algorithm = landmarks_basic
distance_estimator_algorithm = landmarks_sc
num_sampled_pairs = 100

Загрузим граф:

In [12]:
graph = load_graph(graph_path, directed=False)
[edges, nodes, adjacency] = graph
print(f"загружен граф с {len(nodes)} вершинами и {len(edges)} рёбрами")

загружен граф с 18772 вершинами и 396160 рёбрами


Выберем пары случайных вершин:

In [13]:
pairs = []
nodes_list = list(nodes)
for _ in range(num_sampled_pairs):
    s, t = random.sample(nodes_list, 2)
    pairs.append((s, t))

И посчитам: (реальное расстояние, оценка расстояния):

In [14]:
distances_info = []

for s, t in pairs:
    exact_distance = run_bfs_n_return_distances(adjacency, s).get(t, float('inf'))
    estimated_distance = distance_estimator_algorithm(adjacency, nodes, s, t, k)
    distances_info.append((exact_distance, estimated_distance))

После этого посчитаем разные метрики для оценки точности работы алгоритма:

In [15]:
import math

# filter out invalid (inf or nan) values
valid_distances_info = [
    (e, f) for e, f in distances_info
    if math.isfinite(e) and math.isfinite(f)
]

abs_errors = [abs(e - f) for e, f in valid_distances_info]

Посчитаем MAE (Mean Absolute Error). Данная метрика не очень чувствительна к выбросам. Чем она меньше, тем, очевидно, лучше. MAE = 0 - идеальный случай. Также посчитаем минимальную и максимальную ошибки:

In [16]:
min_err = min(abs_errors)
max_err = max(abs_errors)
mae = sum(abs_errors) / len(abs_errors)
print("min error:", min_err)
print("max error:", max_err)
print("MAE:", mae)

min error: 0
max error: 2
MAE: 0.5111111111111111
